# Temperature Gradient Analysis: ΔT vs Source Concentrations

**Motivation**: Does the diurnal temperature range (ΔT = T_max − T_min) relate to source concentrations, and does coloring by daily average temperature reveal a gradient that distinguishes thermal regimes?

**Biomass burning** (charcoal + wood) is expected to increase with cold mornings (heating demand). **Fossil fuel combustion** may respond differently — driven by traffic/industry rather than heating needs, or may also increase if cold weather increases vehicle use.

| Section | Analysis |
|---------|----------|
| **1** | ΔT vs source conc, colored by T_avg (continuous gradient) |
| **2** | ΔT vs source conc, colored by T_min and T_max |
| **3** | T_min vs concentration — does extreme cold drive burning vs fossil fuel? |
| **4** | T_min vs T_max scatter, colored by source conc (thermal regime maps) |

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
notebook_dir = str((repo_root / 'notebooks').resolve())
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

scripts_path = str((repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE
from data_matching import (
    load_aethalometer_data, load_filter_data, add_base_filter_id,
    match_all_parameters, load_etad_factors_with_filter_ids,
)

print(f'Repo root: {repo_root}')

In [ ]:
SEASONS = {
    'Dry Season': [10, 11, 12, 1, 2],
    'Belg Rainy Season': [3, 4, 5],
    'Kiremt Rainy Season': [6, 7, 8, 9],
}
SEASONS_ORDER = ['Dry Season', 'Belg Rainy Season', 'Kiremt Rainy Season']
SEASON_COLORS = {
    'Dry Season': '#E67E22',
    'Belg Rainy Season': '#27AE60',
    'Kiremt Rainy Season': '#3498DB',
}
SEASON_MARKERS = {
    'Dry Season': 'o',
    'Belg Rainy Season': 's',
    'Kiremt Rainy Season': '^',
}

def get_season_3(month):
    for season, months in SEASONS.items():
        if month in months:
            return season
    return None

FACTOR_TO_FRAC = {
    'GF3 (Charcoal)': 'charcoal_frac', 'GF2 (Wood Burning)': 'wood_frac',
    'GF5 (Fossil Fuel Combustion)': 'fossil_fuel_frac',
    'GF4 (Polluted Marine)': 'polluted_marine_frac',
    'GF1 (Sea Salt Mixed)': 'sea_salt_frac',
}
FACTOR_TO_CONC = {
    'K_F3 Charcoal (ug/m3)': 'charcoal_conc',
    'K_F2 Wood Burning (ug/m3)': 'wood_conc',
    'K_F5 Fossil Fuel Combustion (ug/m3)': 'fossil_fuel_conc',
    'K_F4 Polluted Marine (ug/m3)': 'polluted_marine_conc',
    'K_F1 Sea Salt Mixed (ug/m3)': 'sea_salt_conc',
}
frac_cols = list(FACTOR_TO_FRAC.values())
conc_cols = list(FACTOR_TO_CONC.values())

FONT = {'title': 16, 'axis': 14, 'tick': 12, 'legend': 11, 'annot': 11}
plt.rcParams.update({
    'font.size': FONT['tick'], 'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis'], 'legend.fontsize': FONT['legend'],
})

output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'temperature_gradient'
plots_dir = output_root / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
print(f'Outputs: {plots_dir}')

In [ ]:
# === Load & merge ===

def resolve_met_path(*filenames):
    for base in [data_root / 'Weather Data' / 'Meteostat', Path(notebook_dir)]:
        for name in filenames:
            if (base / name).exists():
                return base / name
    raise FileNotFoundError(filenames)

met_daily = pd.read_csv(resolve_met_path(
    'Addis_Ababa_daily_Average_met_Data.csv',
    'Addis Ababa daily Average met Data.csv'), encoding='utf-8-sig')
met_daily['date'] = pd.to_datetime(met_daily['date'])
met_daily['date_only'] = met_daily['date'].dt.normalize()
met_daily['temp_range'] = met_daily['tmax'] - met_daily['tmin']
met_daily['Month'] = met_daily['date'].dt.month
met_daily['season'] = met_daily['Month'].apply(get_season_3)

# Source data
factors_df = load_etad_factors_with_filter_ids()
factors_df = factors_df.rename(columns={**FACTOR_TO_FRAC, **FACTOR_TO_CONC})
aethalometer_data = load_aethalometer_data()
filter_data = load_filter_data()
filter_data = add_base_filter_id(filter_data)
df_aeth = aethalometer_data.get('Addis_Ababa')
bc_df = match_all_parameters('Addis_Ababa', 'ETAD', df_aeth, filter_data)

etad_filters = filter_data[filter_data['Site'] == 'ETAD'][['SampleDate', 'FilterId']].drop_duplicates()
etad_filters = etad_filters.rename(columns={'SampleDate': 'date', 'FilterId': 'base_filter_id'})
bc_df['date'] = pd.to_datetime(bc_df['date'])
etad_filters['date'] = pd.to_datetime(etad_filters['date'])
bc_with_id = pd.merge(bc_df, etad_filters, on='date', how='left')

factor_merge_cols = ['base_filter_id'] + frac_cols + conc_cols
available_merge_cols = [c for c in factor_merge_cols if c in factors_df.columns]
df = pd.merge(bc_with_id, factors_df[available_merge_cols], on='base_filter_id', how='left')
df['Month'] = df['date'].dt.month
df['season'] = df['Month'].apply(get_season_3)
df['date_only'] = df['date'].dt.normalize()

if 'charcoal_conc' in df.columns and 'wood_conc' in df.columns:
    df['biomass_conc'] = df['charcoal_conc'].fillna(0) + df['wood_conc'].fillna(0)

met_merge = met_daily[['date_only', 'tavg', 'tmin', 'tmax', 'temp_range']].copy()
df = pd.merge(df, met_merge, on='date_only', how='left', suffixes=('', '_met'))

# Working subset: non-zero biomass with temperature data
df_t = df.dropna(subset=['temp_range', 'tavg', 'tmin', 'tmax', 'biomass_conc']).copy()
df_t = df_t[df_t['biomass_conc'] > 0]

# Working subset for fossil fuel (may differ in sample count)
df_ff = df.dropna(subset=['temp_range', 'tavg', 'tmin', 'tmax', 'fossil_fuel_conc']).copy()
df_ff = df_ff[df_ff['fossil_fuel_conc'] > 0]

# Combined subset: samples that have BOTH biomass and fossil fuel data
df_both = df.dropna(subset=['temp_range', 'tavg', 'tmin', 'tmax', 'biomass_conc', 'fossil_fuel_conc']).copy()
df_both = df_both[(df_both['biomass_conc'] > 0) & (df_both['fossil_fuel_conc'] > 0)]

print(f'Working dataset (biomass > 0):     {len(df_t)} samples')
print(f'Working dataset (fossil fuel > 0): {len(df_ff)} samples')
print(f'Working dataset (both > 0):        {len(df_both)} samples')
print(f"\n  T_avg: {df_t['tavg'].min():.1f} – {df_t['tavg'].max():.1f} °C")
print(f"  T_min: {df_t['tmin'].min():.1f} – {df_t['tmin'].max():.1f} °C")
print(f"  T_max: {df_t['tmax'].min():.1f} – {df_t['tmax'].max():.1f} °C")
print(f"  ΔT:    {df_t['temp_range'].min():.1f} – {df_t['temp_range'].max():.1f} °C")
print(f"  Biomass:     {df_t['biomass_conc'].min():.3f} – {df_t['biomass_conc'].max():.3f} µg/m³")
print(f"  Fossil fuel: {df_ff['fossil_fuel_conc'].min():.3f} – {df_ff['fossil_fuel_conc'].max():.3f} µg/m³")

---

## Section 1: ΔT vs Source Concentration, colored by T_avg

Continuous color gradient reveals whether the ΔT–concentration relationship is driven by overall cold days (low T_avg) vs warm days. **Top row**: biomass burning sources. **Bottom row**: fossil fuel combustion for comparison.

---

In [ ]:
# =============================================================================
# 1A: ΔT vs Source Concentration, colored by T_avg
# Top row: biomass sources | Bottom row: fossil fuel (+ biomass total for ref)
# =============================================================================

all_cols = [
    ('charcoal_conc', 'Charcoal (\u00b5g/m\u00b3)'),
    ('wood_conc',     'Wood Burning (\u00b5g/m\u00b3)'),
    ('biomass_conc',  'Biomass Total (\u00b5g/m\u00b3)'),
    ('fossil_fuel_conc', 'Fossil Fuel (\u00b5g/m\u00b3)'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (conc_col, conc_label) in enumerate(all_cols):
    row, col = divmod(idx, 2)
    ax = axes[row, col]

    # Use df_t for biomass sources, df_ff for fossil fuel
    if conc_col == 'fossil_fuel_conc':
        src_df = df_ff
    else:
        src_df = df_t
    plot_df = src_df.dropna(subset=[conc_col]).copy()
    plot_df = plot_df[plot_df[conc_col] > 0]

    sc = ax.scatter(plot_df['temp_range'], plot_df[conc_col],
                   c=plot_df['tavg'], cmap='coolwarm', s=55, alpha=0.8,
                   edgecolors='black', linewidths=0.4,
                   vmin=plot_df['tavg'].quantile(0.02),
                   vmax=plot_df['tavg'].quantile(0.98))

    # Regression
    if len(plot_df) >= 5 and plot_df['temp_range'].nunique() > 1:
        slope, intercept, r, p, _ = stats.linregress(plot_df['temp_range'], plot_df[conc_col])
        x_line = np.linspace(plot_df['temp_range'].min(), plot_df['temp_range'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'k-', linewidth=2, alpha=0.7)

        sign = '+' if intercept >= 0 else '-'
        ax.text(0.05, 0.95,
                f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, p = {p:.3f}\nn = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('\u0394T: Diurnal Temperature Range (\u00b0C)', fontsize=FONT['axis'])
    ax.set_ylabel(conc_label, fontsize=FONT['axis'])
    ax.set_title(conc_label.split('(')[0].strip(), fontsize=FONT['title'], fontweight='bold')
    ax.grid(True, alpha=0.3)

    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label('T avg (\u00b0C)', fontsize=FONT['annot'])

plt.suptitle('\u0394T vs Source Concentration \u2014 Colored by Daily Average Temperature\n'
             '(top: biomass burning, bottom-left: biomass total, bottom-right: fossil fuel)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'dt_vs_sources_colored_tavg.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

## Section 2: ΔT vs Source Concentration, colored by T_min and T_max

The cold-morning hypothesis: does T_min (overnight low) drive burning more than the overall temperature range? Compare biomass burning vs fossil fuel combustion response to cold mornings.

---

In [ ]:
# =============================================================================
# 2A: ΔT vs Source Concentration, colored by T_min
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (conc_col, conc_label) in enumerate(all_cols):
    row, col = divmod(idx, 2)
    ax = axes[row, col]

    src_df = df_ff if conc_col == 'fossil_fuel_conc' else df_t
    plot_df = src_df.dropna(subset=[conc_col]).copy()
    plot_df = plot_df[plot_df[conc_col] > 0]

    sc = ax.scatter(plot_df['temp_range'], plot_df[conc_col],
                   c=plot_df['tmin'], cmap='coolwarm', s=55, alpha=0.8,
                   edgecolors='black', linewidths=0.4,
                   vmin=plot_df['tmin'].quantile(0.02),
                   vmax=plot_df['tmin'].quantile(0.98))

    if len(plot_df) >= 5 and plot_df['temp_range'].nunique() > 1:
        slope, intercept, r, p, _ = stats.linregress(plot_df['temp_range'], plot_df[conc_col])
        x_line = np.linspace(plot_df['temp_range'].min(), plot_df['temp_range'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'k-', linewidth=2, alpha=0.7)

        sign = '+' if intercept >= 0 else '-'
        ax.text(0.05, 0.95,
                f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, p = {p:.3f}\nn = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('\u0394T: Diurnal Temperature Range (\u00b0C)', fontsize=FONT['axis'])
    ax.set_ylabel(conc_label, fontsize=FONT['axis'])
    ax.set_title(conc_label.split('(')[0].strip(), fontsize=FONT['title'], fontweight='bold')
    ax.grid(True, alpha=0.3)

    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label('T min (\u00b0C)', fontsize=FONT['annot'])

plt.suptitle('\u0394T vs Source Concentration \u2014 Colored by Overnight Minimum Temperature\n'
             '(top: biomass burning, bottom-left: biomass total, bottom-right: fossil fuel)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'dt_vs_sources_colored_tmin.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 2B: ΔT vs Source Concentration, colored by T_max
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (conc_col, conc_label) in enumerate(all_cols):
    row, col = divmod(idx, 2)
    ax = axes[row, col]

    src_df = df_ff if conc_col == 'fossil_fuel_conc' else df_t
    plot_df = src_df.dropna(subset=[conc_col]).copy()
    plot_df = plot_df[plot_df[conc_col] > 0]

    sc = ax.scatter(plot_df['temp_range'], plot_df[conc_col],
                   c=plot_df['tmax'], cmap='coolwarm', s=55, alpha=0.8,
                   edgecolors='black', linewidths=0.4,
                   vmin=plot_df['tmax'].quantile(0.02),
                   vmax=plot_df['tmax'].quantile(0.98))

    if len(plot_df) >= 5 and plot_df['temp_range'].nunique() > 1:
        slope, intercept, r, p, _ = stats.linregress(plot_df['temp_range'], plot_df[conc_col])
        x_line = np.linspace(plot_df['temp_range'].min(), plot_df['temp_range'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'k-', linewidth=2, alpha=0.7)

        sign = '+' if intercept >= 0 else '-'
        ax.text(0.05, 0.95,
                f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, p = {p:.3f}\nn = {len(plot_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('\u0394T: Diurnal Temperature Range (\u00b0C)', fontsize=FONT['axis'])
    ax.set_ylabel(conc_label, fontsize=FONT['axis'])
    ax.set_title(conc_label.split('(')[0].strip(), fontsize=FONT['title'], fontweight='bold')
    ax.grid(True, alpha=0.3)

    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label('T max (\u00b0C)', fontsize=FONT['annot'])

plt.suptitle('\u0394T vs Source Concentration \u2014 Colored by Afternoon Maximum Temperature\n'
             '(top: biomass burning, bottom-left: biomass total, bottom-right: fossil fuel)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'dt_vs_sources_colored_tmax.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

## Section 3: T_min as a Driver — Biomass vs Fossil Fuel

Hypothesis: **extremely cold mornings** trigger charcoal/wood burning for heating. If true, T_min should be a stronger predictor of biomass concentration than T_avg or T_max. Fossil fuel combustion (traffic, industry) may show a different or weaker temperature response.

---

In [ ]:
# =============================================================================
# 3A: T_min, T_avg, T_max vs Biomass AND Fossil Fuel — head-to-head
# Top row: Biomass Total | Bottom row: Fossil Fuel Combustion
# =============================================================================

temp_predictors = [
    ('tmin', 'T min (\u00b0C)', '#2980B9'),
    ('tavg', 'T avg (\u00b0C)', '#8E44AD'),
    ('tmax', 'T max (\u00b0C)', '#E74C3C'),
]

source_rows = [
    ('biomass_conc',     'Biomass Total (\u00b5g/m\u00b3)', df_t),
    ('fossil_fuel_conc', 'Fossil Fuel (\u00b5g/m\u00b3)',   df_ff),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for row_idx, (conc_col, conc_label, src_df) in enumerate(source_rows):
    for col_idx, (t_col, t_label, clr) in enumerate(temp_predictors):
        ax = axes[row_idx, col_idx]
        plot_df = src_df.dropna(subset=[t_col, conc_col]).copy()

        # Color by season using different markers
        for season in SEASONS_ORDER:
            mask = plot_df['season'] == season
            subset = plot_df[mask]
            if len(subset) > 0:
                ax.scatter(subset[t_col], subset[conc_col],
                          c=SEASON_COLORS[season], marker=SEASON_MARKERS[season],
                          s=45, alpha=0.7, edgecolors='black', linewidths=0.3,
                          label=season)

        # Overall regression
        if len(plot_df) >= 5 and plot_df[t_col].nunique() > 1:
            slope, intercept, r, p, _ = stats.linregress(plot_df[t_col], plot_df[conc_col])
            x_line = np.linspace(plot_df[t_col].min(), plot_df[t_col].max(), 100)
            ax.plot(x_line, slope * x_line + intercept, color=clr, linewidth=2.5, alpha=0.8)

            # Per-season regressions
            for season in SEASONS_ORDER:
                s_df = plot_df[plot_df['season'] == season]
                if len(s_df) >= 5 and s_df[t_col].nunique() > 1:
                    sl, intc, r_s, p_s, _ = stats.linregress(s_df[t_col], s_df[conc_col])
                    x_s = np.linspace(s_df[t_col].min(), s_df[t_col].max(), 50)
                    ax.plot(x_s, sl * x_s + intc, color=SEASON_COLORS[season],
                           linewidth=1.5, linestyle='--', alpha=0.7)

            sign = '+' if intercept >= 0 else '-'
            ax.text(0.05, 0.95,
                    f'Overall: y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                    f'R\u00b2 = {r**2:.3f}, p = {p:.3f}\nn = {len(plot_df)}',
                    transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_xlabel(t_label, fontsize=FONT['axis'])
        ax.set_ylabel(conc_label if col_idx == 0 else '', fontsize=FONT['axis'])
        ax.set_title(f'{t_label} \u2192 {conc_label.split("(")[0].strip()}',
                     fontsize=FONT['title'] - 1, fontweight='bold', color=clr)
        ax.grid(True, alpha=0.3)
        if col_idx == 2 and row_idx == 0:
            ax.legend(fontsize=FONT['legend'] - 1, loc='upper right')

plt.suptitle('Which Temperature Metric Best Predicts Each Source?\n'
             'Top: Biomass Burning | Bottom: Fossil Fuel Combustion\n'
             '(solid = overall regression, dashed = per-season)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(str(plots_dir / 'temp_predictors_biomass_vs_fossil.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Summary comparison table
print('\n' + '=' * 80)
print('PREDICTOR COMPARISON: Temperature metrics vs Biomass AND Fossil Fuel')
print('=' * 80)
print(f'\n{"Predictor":<12s} {"Biomass R\u00b2":>10s} '
      f'{"Fossil R\u00b2":>10s} {"Stronger?":>12s}')
print('-' * 50)

for t_col, t_label, _ in temp_predictors:
    # Biomass
    valid_b = df_t.dropna(subset=[t_col, 'biomass_conc'])
    r_b, p_b = stats.pearsonr(valid_b[t_col], valid_b['biomass_conc'])
    # Fossil fuel
    valid_f = df_ff.dropna(subset=[t_col, 'fossil_fuel_conc'])
    r_f, p_f = stats.pearsonr(valid_f[t_col], valid_f['fossil_fuel_conc'])
    stronger = 'Biomass' if abs(r_b) > abs(r_f) else 'Fossil'
    print(f'{t_label:<12s} {r_b**2:10.3f} {r_f**2:10.3f} {stronger:>12s}')

# Also deltaT
valid_b = df_t.dropna(subset=['temp_range', 'biomass_conc'])
r_b, p_b = stats.pearsonr(valid_b['temp_range'], valid_b['biomass_conc'])
valid_f = df_ff.dropna(subset=['temp_range', 'fossil_fuel_conc'])
r_f, p_f = stats.pearsonr(valid_f['temp_range'], valid_f['fossil_fuel_conc'])
stronger = 'Biomass' if abs(r_b) > abs(r_f) else 'Fossil'
print(f'{"\u0394T":<12s} {r_b**2:10.3f} {r_f**2:10.3f} {stronger:>12s}')

In [ ]:
# =============================================================================
# 3B: Source concentration binned by T_min quartile — Biomass vs Fossil Fuel
# =============================================================================

# Compute quartiles on the combined dataset
df_t['tmin_quartile'] = pd.qcut(df_t['tmin'], q=4, duplicates='drop')
df_ff['tmin_quartile'] = pd.qcut(df_ff['tmin'], q=4, duplicates='drop')

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

source_info = [
    (df_t,  'biomass_conc',     'Biomass Total', 'YlOrRd'),
    (df_ff, 'fossil_fuel_conc', 'Fossil Fuel',   'PuBu'),
]

for row_idx, (src_df, conc_col, src_name, cmap_name) in enumerate(source_info):
    quartiles = sorted(src_df['tmin_quartile'].dropna().unique())

    # Box plot
    ax = axes[row_idx, 0]
    box_data = [src_df[src_df['tmin_quartile'] == q][conc_col].dropna() for q in quartiles]
    bp = ax.boxplot(box_data, labels=[f'Q{i+1}' for i in range(len(quartiles))],
                    patch_artist=True, widths=0.6,
                    medianprops=dict(color='red', linewidth=2))

    cold_warm = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(quartiles)))
    for patch, color in zip(bp['boxes'], cold_warm):
        patch.set_facecolor(color)

    for i, q in enumerate(quartiles):
        n = len(box_data[i])
        ax.text(i + 1, ax.get_ylim()[0] - 0.04 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
                f'{q}\nn={n}', ha='center', fontsize=FONT['annot'] - 2, va='top')

    ax.set_ylabel(f'{src_name} Conc (µg/m³)', fontsize=FONT['axis'])
    ax.set_title(f'{src_name} by T_min Quartile\n(Q1 = coldest mornings)', fontsize=FONT['title'])
    ax.grid(True, axis='y', alpha=0.3)

    # Mean + SEM bar chart with season breakdown
    ax = axes[row_idx, 1]
    x = np.arange(len(quartiles))
    width = 0.25
    for i, season in enumerate(SEASONS_ORDER):
        means, sems, counts = [], [], []
        for q in quartiles:
            subset = src_df[(src_df['tmin_quartile'] == q) & (src_df['season'] == season)][conc_col]
            means.append(subset.mean() if len(subset) > 0 else 0)
            sems.append(subset.sem() if len(subset) > 1 else 0)
            counts.append(len(subset))
        bars = ax.bar(x + i * width - width, means, width, yerr=sems,
                      label=season, color=SEASON_COLORS[season],
                      edgecolor='black', capsize=3)
        for j, (b, n) in enumerate(zip(bars, counts)):
            if n > 0:
                ax.text(b.get_x() + b.get_width() / 2, b.get_height() + sems[j] + 0.002,
                        f'{n}', ha='center', fontsize=7, color=SEASON_COLORS[season])

    ax.set_xticks(x)
    ax.set_xticklabels([f'Q{i+1}' for i in range(len(quartiles))])
    ax.set_ylabel(f'Mean {src_name} Conc (µg/m³)', fontsize=FONT['axis'])
    ax.set_title(f'{src_name} by T_min Quartile × Season', fontsize=FONT['title'])
    ax.legend(fontsize=FONT['legend'] - 1)
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Does Extreme Cold (Low T_min) Drive Higher Concentrations?\n'
             'Top: Biomass Burning | Bottom: Fossil Fuel Combustion',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'tmin_quartile_biomass_vs_fossil.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

## Section 4: Thermal Regime Maps (T_min vs T_max)

Scatter of T_min vs T_max colored by source concentration. This separates thermal regimes:

- **Bottom-left**: cold all day → expect high biomass (heating demand)
- **Bottom-right**: cold morning, warm afternoon (large ΔT) → morning-only burning?
- **Top-right**: warm all day → low biomass, but fossil fuel may persist

Diagonal lines show constant ΔT. Compare biomass vs fossil fuel spatial patterns.

---

In [ ]:
# =============================================================================
# 4A: Thermal Regime Map — T_min vs T_max
# Top row: colored by source concentration (biomass vs fossil fuel)
# Bottom row: colored by season, sized by source concentration
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

regime_sources = [
    (df_t,  'biomass_conc',     'Biomass Total (µg/m³)', 'YlOrRd'),
    (df_ff, 'fossil_fuel_conc', 'Fossil Fuel (µg/m³)',   'YlGnBu'),
]

def add_dt_diagonals(ax, tmin_range):
    """Add constant ΔT diagonal lines."""
    for dt_val in [5, 10, 15, 20]:
        t_range = np.linspace(tmin_range[0] - 2, tmin_range[1] + 2, 50)
        ax.plot(t_range, t_range + dt_val, '--', color='gray', alpha=0.4, linewidth=1)
        mid = len(t_range) // 2
        ax.text(t_range[mid], t_range[mid] + dt_val + 0.3,
                f'ΔT={dt_val}°', fontsize=8, color='gray', alpha=0.6,
                rotation=45, ha='center')

# Top row: colored by concentration
for col_idx, (src_df, conc_col, conc_label, cmap_name) in enumerate(regime_sources):
    ax = axes[0, col_idx]
    plot_df = src_df.dropna(subset=['tmin', 'tmax', conc_col]).copy()

    sc = ax.scatter(plot_df['tmin'], plot_df['tmax'],
                   c=plot_df[conc_col], cmap=cmap_name, s=55, alpha=0.8,
                   edgecolors='black', linewidths=0.4,
                   vmin=0, vmax=plot_df[conc_col].quantile(0.95))

    add_dt_diagonals(ax, (plot_df['tmin'].min(), plot_df['tmin'].max()))

    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label(conc_label, fontsize=FONT['annot'])
    ax.set_xlabel('T min (°C)', fontsize=FONT['axis'])
    ax.set_ylabel('T max (°C)', fontsize=FONT['axis'])
    ax.set_title(f'Thermal Regime — Color = {conc_label.split("(")[0].strip()}',
                 fontsize=FONT['title'] - 1)
    ax.grid(True, alpha=0.2)

# Bottom row: colored by season, sized by concentration
for col_idx, (src_df, conc_col, conc_label, _) in enumerate(regime_sources):
    ax = axes[1, col_idx]
    plot_df = src_df.dropna(subset=['tmin', 'tmax', conc_col]).copy()
    size_scale = plot_df[conc_col] / plot_df[conc_col].max() * 200 + 15

    for season in SEASONS_ORDER:
        mask = plot_df['season'] == season
        subset = plot_df[mask]
        if len(subset) == 0:
            continue
        sizes = size_scale[mask]
        ax.scatter(subset['tmin'], subset['tmax'],
                  c=SEASON_COLORS[season], marker=SEASON_MARKERS[season],
                  s=sizes, alpha=0.7, edgecolors='black', linewidths=0.3,
                  label=season)

    add_dt_diagonals(ax, (plot_df['tmin'].min(), plot_df['tmin'].max()))

    ax.set_xlabel('T min (°C)', fontsize=FONT['axis'])
    ax.set_ylabel('T max (°C)', fontsize=FONT['axis'])
    src_short = conc_label.split('(')[0].strip()
    ax.set_title(f'Thermal Regime — Season, Size = {src_short}',
                 fontsize=FONT['title'] - 1)
    ax.grid(True, alpha=0.2)

    # Combined legend: season + size
    handles, labels = ax.get_legend_handles_labels()
    for bm_val in [0.05, 0.15, 0.30]:
        sz = bm_val / plot_df[conc_col].max() * 200 + 15
        handles.append(ax.scatter([], [], s=sz, c='gray', edgecolors='black', linewidths=0.3))
        labels.append(f'{bm_val:.2f} µg/m³')
    ax.legend(handles, labels, fontsize=FONT['legend'] - 1, loc='upper left', ncol=2,
              title='Season / Size', title_fontsize=FONT['legend'] - 1)

plt.suptitle('Thermal Regime Maps: T_min vs T_max\n'
             'Left: Biomass Burning | Right: Fossil Fuel Combustion\n'
             '(diagonal lines = constant ΔT; bottom-left = cold all day)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(str(plots_dir / 'thermal_regime_map_biomass_vs_fossil.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Summary: correlation of each thermal metric with each source
# =============================================================================

print('=' * 90)
print('CORRELATION MATRIX: Temperature Metrics vs ALL Sources')
print('=' * 90)

t_metrics = ['tmin', 'tavg', 'tmax', 'temp_range']
b_cols = ['charcoal_conc', 'wood_conc', 'biomass_conc', 'fossil_fuel_conc']
t_labels = ['T min', 'T avg', 'T max', 'ΔT']
b_labels = ['Charcoal', 'Wood', 'Biomass Total', 'Fossil Fuel']

print(f'\n{"":<12s}', end='')
for bl in b_labels:
    print(f'  {bl:>18s}', end='')
print()
print('-' * 90)

for tl, tc in zip(t_labels, t_metrics):
    print(f'{tl:<12s}', end='')
    for bc in b_cols:
        # Use appropriate working subset
        if bc == 'fossil_fuel_conc':
            valid = df_ff.dropna(subset=[tc, bc])
        else:
            valid = df_t.dropna(subset=[tc, bc])
        valid = valid[valid[bc] > 0]
        if len(valid) >= 5:
            r, p = stats.pearsonr(valid[tc], valid[bc])
            sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
            print(f'  {r:6.3f} (p={p:.3f}){sig:>2s}', end='')
        else:
            print(f'  {"n/a":>18s}', end='')
    print()

print('\n** = p < 0.01, * = p < 0.05')
print('\nInterpretation:')
print('  Negative r with T_min = colder mornings → more of that source')
print('  Positive r with ΔT   = wider range (clear sky) → more')
print('  Compare |r| across metrics to identify strongest driver')
print('  Compare biomass vs fossil fuel to see if temperature drives them differently')